# PennyLane VQE

Minimize a two-qubit Hamiltonian with parameter-shift gradients and compare the complete energy trace.

The SDK reference and MettleQ calls below use the same circuit and result contract. Timing includes the complete call shown.

In [ ]:
import numpy as np
import pennylane as qml
from pennylane import numpy as pnp

from mettleq.integrations.pennylane import MettleQDevice
from tutorials._support import (
    benchmark,
    emit_result,
    max_abs_error,
    pennylane_selection,
    phase_aligned_statevector_error,
    total_variation_distance,
)

In [ ]:
hamiltonian = -1.05 * qml.I(0) + 0.39 * qml.Z(0) - 0.39 * qml.Z(1) - 0.01 * (qml.Z(0) @ qml.Z(1)) + 0.18 * (qml.X(0) @ qml.X(1))

def make_energy(device):
    @qml.qnode(device, diff_method="parameter-shift")
    def energy(weights):
        qml.RY(weights[0], wires=0)
        qml.CNOT(wires=[0, 1])
        qml.RY(weights[1], wires=1)
        return qml.expval(hamiltonian)
    return energy

def optimize(energy):
    weights = pnp.array([0.2, -0.3], requires_grad=True)
    trace = []
    for _ in range(10):
        trace.append(float(energy(weights)))
        weights = weights - 0.12 * qml.grad(energy)(weights)
    trace.append(float(energy(weights)))
    return np.asarray(trace)

reference_energy = make_energy(qml.device("default.qubit", wires=2))
reference, reference_ms, _ = benchmark(lambda: optimize(reference_energy), repeats=2)
mettleq_device = MettleQDevice(wires=2, method="statevector", device="cpu")
mettleq_energy = make_energy(mettleq_device)
candidate, mettleq_ms, _ = benchmark(lambda: optimize(mettleq_energy), repeats=2)
error = max_abs_error(reference, candidate)
method, device = pennylane_selection(mettleq_device)
tutorial_result = emit_result(
    notebook="pennylane/05_vqe.ipynb",
    framework="pennylane",
    reference_ms=reference_ms,
    mettleq_ms=mettleq_ms,
    check="VQE energy trace atol=5e-5",
    passed=error <= 5e-5 and candidate[-1] < candidate[0],
    exact_match=bool(np.array_equal(reference, candidate)),
    selected_method=method,
    selected_device=device,
    metrics={"max_energy_error": error, "reference_final": reference[-1], "mettleq_final": candidate[-1]},
)